Are there duplicates? what makes a report unique, given there is not ID? the detective query in...

In [0]:
%sql
SELECT *
FROM eindhoven.bronze.meldingen
LIMIT 10;

In [0]:
%sql
SELECT bron, onderwerp, subonderwerp, aangemaakt, locatie, COUNT(*) AS aantal
FROM eindhoven.bronze.meldingen
GROUP BY bron, onderwerp, subonderwerp, aangemaakt, locatie
HAVING COUNT(*) > 1
ORDER BY aantal DESC
LIMIT 20;

In [0]:
%sql
SELECT *
FROM eindhoven.bronze.meldingen
WHERE locatie = 'Vredeoord 201, 5621 CX Eindhoven, Nederland'
AND
aangemaakt = '2025-02-24T18:40:54.000+00:00';

In [0]:
%sql
SELECT
-- ws = "with seperator"; the | keeps "ab"+"c" different from "a"+"bc".
COUNT(DISTINCT(sha2(concat_ws('|', bron, onderwerp, subonderwerp, aangemaakt, opgelost, locatie), 256))) AS melding_id
FROM eindhoven.bronze.meldingen;


In [0]:
%sql
SELECT 
    sha2(concat_ws('|', bron, onderwerp, subonderwerp, aangemaakt, opgelost, locatie), 256) AS melding_id, 
    bron, 
    onderwerp, 
    subonderwerp, 
    aangemaakt, 
    opgelost, 
    datediff(opgelost, aangemaakt) AS oplostijd_dagen, 
    buurtnaam, 
    buurtcode, 
    wijkcode, 
    wijknaam, 
    stadsdeelcode, 
    stadsdeelnaam, 
    locatie, 
    x_coord, 
    y_coord, 
    geo_point_2d
FROM eindhoven.bronze.meldingen
WHERE opgelost >= aangemaakt 
AND 1 = 0;

--


In [0]:
%sql


SELECT COUNT(*) 
FROM eindhoven.bronze.meldingen
WHERE opgelost IS NULL;

In [0]:
%sql
SELECT 
    CASE 
        WHEN timestampdiff(minute, opgelost, aangemaakt) < 60 THEN 'verschil < 1 uur'
        WHEN timestampdiff(minute, opgelost, aangemaakt) < 1440 THEN 'verschil 1 uur tot 1 dag'
        WHEN timestampdiff(minute, opgelost, aangemaakt) >= 1440 THEN 'verschil > 1 dag'
    END AS oplostijd_categorie,
    COUNT(*) AS aantal
FROM eindhoven.bronze.meldingen
WHERE opgelost < aangemaakt
AND timestampdiff(minute, opgelost, aangemaakt) BETWEEN 60 AND 1440
GROUP BY oplostijd_categorie
ORDER BY aantal DESC
LIMIT 10;

In [0]:
%sql
SELECT
    timestampdiff(minute, opgelost, aangemaakt) AS verschil_minuten,
    COUNT(*) AS aantal
FROM    eindhoven.bronze.meldingen
WHERE opgelost < aangemaakt
GROUP BY verschil_minuten
ORDER BY aantal DESC